# SınıfKart AI · Derleme ve kart testleri
Drive'daki SinifKart_Calisma.zip güncel kaynak paketidir. İlk hücre Google hesabınızdan Drive bağlama izni ister. Geçici çalışma klasöründe proje açılır. Test sonuçları hücre çıktılarında görünür.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, tempfile, subprocess, os
drive.mount('/content/drive')
archive = Path('/content/drive/MyDrive/PLİCKERS/SinifKart_Calisma.zip')
workspace = Path(tempfile.mkdtemp(prefix='sinifkart-'))
with zipfile.ZipFile(archive) as bundle:
    for name in bundle.namelist():
        if not (workspace / name).resolve().is_relative_to(workspace):
            raise ValueError(name)
    bundle.extractall(workspace)
root = workspace / 'hibrit'
def run(command,cwd=root):
    subprocess.run(command,cwd=cwd,check=True)

In [ ]:
import time, urllib.request
web = root/'web'
run(['npm','install','--no-audit','--no-fund'], web)
run(['npm','run','build'], web)
assert (web/'dist/index.html').is_file()
server = subprocess.Popen(['node','node_modules/vite/bin/vite.js','--host','127.0.0.1','--port','5173','--strictPort'],cwd=web)
try:
    for attempt in range(40):
        if server.poll() is not None: raise RuntimeError('Vite durdu')
        try:
            with urllib.request.urlopen('http://127.0.0.1:5173/',timeout=1) as response:
                assert b'/src/main.tsx' in response.read()
            break
        except OSError:
            time.sleep(.25)
    else: raise RuntimeError('Vite yanıt vermedi')
    print('Vite geliştirme sunucusu ve üretim derlemesi geçti.')
finally:
    server.terminate()
    server.wait()

In [ ]:
import sys
run([sys.executable,'-m','venv',str(workspace/'env')])
python=str(workspace/'env/bin/python')
run([python,'-m','pip','install','-r','reference/requirements.txt','reportlab','pypdfium2','Pillow'])
for script in ['cards.py','test_detection.py','test_session.py','test_classroom.py']:
    run([python,str(root/'reference'/script)])